# Import data and packages

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

from SNPs_selection.snps_selection import run_bootstrap_single

In [2]:
DATA_DIR = PROJECT_ROOT / "data"

data_path = DATA_DIR / "synth_data.xlsx"

data = pd.read_excel(data_path)

SNPs_list = ['SNP_1', 'SNP_2', 'SNP_3', 'SNP_4']

# Run bootstrap ranking

In [3]:
B = 1000

bootstrap_U = Parallel(n_jobs=-1, verbose=0)(
    delayed(run_bootstrap_single)(data, SNPs_list, "outcome_U", b) for b in range(B)
)

bootstrap_I = Parallel(n_jobs=-1, verbose=0)(
    delayed(run_bootstrap_single)(data, SNPs_list, "outcome_I", b) for b in range(B)
)


# Visualize the results

In [4]:
bootstrap_dict = {
    "U": bootstrap_U,
    "I": bootstrap_I,
}

snps_ordered_U = []
snps_ordered_I = []

for key in bootstrap_dict.keys():
    rank_dict = defaultdict(list)

    for boot in bootstrap_dict[key]:
        for snp, rank in boot:
            rank_dict[snp].append(rank)

    ranking_score = {snp: 1 - (np.mean(rank_dict[snp]) - 1)/(len(SNPs_list)-1) for snp in rank_dict.keys()}
    
    ordered_snps = sorted(ranking_score.items(), key=lambda x: -x[1])

    if key == "U":
        snps_ordered_U = [snp for snp, _ in ordered_snps]
    elif key == "I":
        snps_ordered_I = [snp for snp, _ in ordered_snps]

    print(f"\nRanking {key}:")
    print(f"{'SNP':<20} {'Ranking Score':>10}")
    for snp, mr in ordered_snps:
        print(f"{snp:<20} {mr:>10.2f}")


Ranking U:
SNP                  Ranking Score
SNP_2                      0.62
SNP_1                      0.60
SNP_3                      0.40
SNP_4                      0.37

Ranking I:
SNP                  Ranking Score
SNP_3                      0.66
SNP_4                      0.48
SNP_1                      0.44
SNP_2                      0.43
